In [1]:
import torch, torchvision, sys
print("torch", torch.__version__, "| tv", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 → Accelerator 확인")

torch 2.10.0+cu128 | tv 0.25.0+cu128
GPU: Tesla T4


In [2]:
import glob, os, shutil, zipfile
DST = '/kaggle/working/samlab'; os.makedirs(DST, exist_ok=True)
zips   = glob.glob('/kaggle/input/**/*.zip', recursive=True)
marker = glob.glob('/kaggle/input/**/train.py', recursive=True)
if zips:
    zipfile.ZipFile(zips[0]).extractall(DST); print("unzipped:", zips[0])
elif marker:
    shutil.copytree(os.path.dirname(marker[0]), DST, dirs_exist_ok=True); print("copied")
else:
    raise SystemExit("Input에 데이터셋이 안 붙었습니다")
os.chdir(DST); print(sorted(os.listdir('.')))

copied
['README.md', 'requirements.txt', 'sam_lab', 'tests', 'train.py']


In [3]:
!python tests/smoke_test.py


--- test_schedule
[  ok  ] warmup rises from ~0
[  ok  ] stable phase is 1.0
[  ok  ] decay ends near 0
[  ok  ] SAM off before decay
[  ok  ] SAM on at decay start
[  ok  ] SAM decay window is 20%  -- start=800
[  ok  ] sam_mode=off never fires
[  ok  ] sam_mode=full always fires

--- test_newton_schulz
[  ok  ] newton_schulz5 compresses the spectrum  -- spread 4.92 -> 1.53
[  ok  ] newton_schulz5 singular values land near 1  -- sv in [0.682, 1.042]
[  ok  ] newton_schulz5 keeps shape (wide)
[  ok  ] newton_schulz5 output is finite

--- test_sam_restores_exactly
[  ok  ] perturb() succeeds
[  ok  ] perturb actually moves weights  -- max move 1.47e-02
[  ok  ] restore() is exact  -- residual 0.00e+00

--- test_sam_rho_scales
[  ok  ] ||eps|| == rho (L2 mode)  -- ratio 10.0000
[  ok  ] ||eps|| equals rho exactly  -- got 0.100000

--- test_sam_handles_nonfinite
[  ok  ] non-finite grads -> perturb refuses
[  ok  ] skip counter increments

--- test_sam_spectral_mode
[  ok  ] spectral mod

In [4]:
!python train.py --optimizer muon --sam_mode decay --epochs 3 --exp sanity \
    --out_dir /kaggle/working/runs --num_workers 2

100%|█████████████████████████████████████████| 169M/169M [27:13<00:00, 103kB/s]

=== sanity-cifar100-resnet18-muon-sam_decay-s0  [3ccc88d5dac3] ===
    device=cuda  params=11.22M (matrix 11.21M)
    optimizer=muon lr=0.001 sam=decay(rho=0.05,l2)
    steps/epoch=195 epochs=3
[sanity-cifar100-resnet18-muon-sam_decay-s0] ep   1/3 train 4.204/7.9% val 3.702/14.3% sam 0.00 28s
[sanity-cifar100-resnet18-muon-sam_decay-s0] ep   2/3 train 3.430/18.9% val 3.157/23.4% sam 0.00 22s
[sanity-cifar100-resnet18-muon-sam_decay-s0] ep   3/3 train 2.975/26.9% val 2.830/29.7% sam 0.60 29s
    -> {"finished": true, "epochs_done": 3, "val_acc": 0.297, "best_val_acc": 0.297, "cumulative_seconds": 79.28436350822449, "sam_skipped": 0}


In [5]:
!python train.py --grid --out_dir /kaggle/working/runs --num_workers 2 --max_seconds 28000


=== s0-cifar100-resnet18-adamw-sam_off-s0  [e304bace0c19] ===
    device=cuda  params=11.22M (matrix 11.21M)
    optimizer=adamw lr=0.001 sam=off
    steps/epoch=195 epochs=30
[s0-cifar100-resnet18-adamw-sam_off-s0] ep   1/30 train 3.870/11.2% val 3.597/16.4% sam 0.00 16s
[s0-cifar100-resnet18-adamw-sam_off-s0] ep   2/30 train 3.034/24.1% val 2.958/27.4% sam 0.00 12s
[s0-cifar100-resnet18-adamw-sam_off-s0] ep   3/30 train 2.437/35.9% val 2.477/36.8% sam 0.00 12s
[s0-cifar100-resnet18-adamw-sam_off-s0] ep   4/30 train 2.054/44.1% val 2.280/41.2% sam 0.00 12s
[s0-cifar100-resnet18-adamw-sam_off-s0] ep   5/30 train 1.783/50.5% val 2.014/47.0% sam 0.00 12s
[s0-cifar100-resnet18-adamw-sam_off-s0] ep   6/30 train 1.594/54.8% val 1.919/49.5% sam 0.00 12s
[s0-cifar100-resnet18-adamw-sam_off-s0] ep   7/30 train 1.437/58.8% val 1.881/50.7% sam 0.00 12s
[s0-cifar100-resnet18-adamw-sam_off-s0] ep   8/30 train 1.315/62.2% val 1.851/52.0% sam 0.00 12s
[s0-cifar100-resnet18-adamw-sam_off-s0] ep   9/

In [6]:
import pandas as pd, glob
rows = []
for f in sorted(glob.glob('/kaggle/working/runs/s0-*/epochs.csv')):
    df = pd.read_csv(f); last = df.iloc[-1]
    sam_ep = df[df.sam_on > 0].epoch_seconds.mean()
    no_sam = df[df.sam_on == 0].epoch_seconds.mean()
    rows.append(dict(run=f.split('/')[-2].replace('s0-cifar100-resnet18-',''),
                     best_acc=round(df.val_acc.max()*100, 2),
                     final_acc=round(last.val_acc*100, 2),
                     skipped=int(last.sam_skipped),
                     min=round(last.cumulative_seconds/60, 1),
                     sam_cost=round(sam_ep/no_sam, 2) if sam_ep==sam_ep else None))
print(pd.DataFrame(rows).to_string(index=False))

               run  best_acc  final_acc  skipped  min  sam_cost
adamw-sam_decay-s0     72.38      72.27        0  7.4      1.98
  adamw-sam_off-s0     72.65      72.51        0  6.2       NaN
 lion-sam_decay-s0     71.63      71.63        0  7.6      1.94
   lion-sam_off-s0     71.59      71.59        0  6.4       NaN
 muon-sam_decay-s0     76.79      76.79        0 13.1      1.50
   muon-sam_off-s0     76.45      76.45        0 11.9       NaN
